### Maximilian Kempa LAB | Programming Foundation Exercises

This simulator creates and updates different smart home sensors with random readings.
Each reading is checked against security, safety, and comfort rules depending on the current system mode.
If an abnormal condition is detected, the system displays an alert and logs the event.

In [129]:
import random
import datetime
import time

In [130]:
def create_sensor(id, location, type):
    return {
        "id": id,
        "location": location,
        "type": type,
        "current_value": None
    }


def create_alert(sensor_id, alert_type, message):
    return {
        "sensor_id": sensor_id,
        "alert_type": alert_type,
        "message": message
    }


sensor_data = [
    create_sensor(1, "Living Room", "Motion"),
    create_sensor(2, "Front Door", "Door"),
    create_sensor(3, "Kitchen", "Temperature"),
    create_sensor(4, "Bedroom", "Smoke")
]

alerts = []

print(sensor_data)

[{'id': 1, 'location': 'Living Room', 'type': 'Motion', 'current_value': None}, {'id': 2, 'location': 'Front Door', 'type': 'Door', 'current_value': None}, {'id': 3, 'location': 'Kitchen', 'type': 'Temperature', 'current_value': None}, {'id': 4, 'location': 'Bedroom', 'type': 'Smoke', 'current_value': None}]


In [131]:
class Sensor:
    def __init__(self, id, location, type):
        self.id = id
        self.location = location
        self.type = type
        self.current_value = None
        self.open_since = None

    def update_value(self, value):
        self.current_value = value

    def read(self):
        return self.current_value

    def isAbnormal(self):
        if self.type == "Temperature":
            return self.current_value < 35 or self.current_value > 95

        elif self.type == "Smoke":
            return self.current_value

        return False

    def reset(self):
        self.current_value = None
        self.open_since = None

In [132]:
motion_sensor = Sensor(id=1, location="Living Room", type="Motion")
door_sensor = Sensor(id=2, location="Front Door", type="Door")
temperature_sensor = Sensor(id=3, location="Kitchen", type="Temperature")
smoke_sensor = Sensor(id=4, location="Bedroom", type="Smoke")

sensors = [motion_sensor, door_sensor, temperature_sensor, smoke_sensor]

In [133]:
def generate_reading(sensor):

    if sensor.type == "Temperature":
        return random.randint(30, 100)

    elif sensor.type == "Motion":
        return random.random() < 0.3

    elif sensor.type == "Door":
        return "OPENED" if random.random() < 0.3 else "CLOSED"

    elif sensor.type == "Smoke":
        return random.random() < 0.05

In [134]:
system_mode = "AWAY"

In [135]:
event_log = []

def log_event(message):
    current_time = datetime.datetime.now().strftime("%H:%M:%S")
    log_message = f"[{current_time}] {message}"
    event_log.append(log_message)
    print(f"[LOG] {log_message}")

In [136]:
def trigger_alert(message):
    print(f"[ALERT!] 🚨 {message}")
    log_event("Sending notification to homeowner...")

In [137]:
def process_reading(sensor, mode):

    if sensor.type == "Smoke" and sensor.current_value:
        return f"CRITICAL: Smoke detected in {sensor.location}!"

    elif sensor.type == "Temperature":
        if sensor.current_value < 35 or sensor.current_value > 95:
            return f"WARNING: Abnormal temperature in {sensor.location}!"

    elif sensor.type == "Motion" and sensor.current_value:
        if mode == "AWAY":
            return f"HIGH SECURITY: Motion detected in {sensor.location} while system is in {mode} mode!"
        elif mode == "SLEEP":
            return f"MEDIUM SECURITY: Motion detected in {sensor.location} while system is in {mode} mode!"

    elif sensor.type == "Door" and sensor.current_value == "OPENED":
        if mode == "AWAY":
            return f"HIGH SECURITY: {sensor.location} opened while system is in {mode} mode!"
        elif mode == "SLEEP":
            return f"MEDIUM SECURITY: {sensor.location} opened while system is in {mode} mode!"

    return None

In [138]:
def sensor_update(sensor, mode):

    if sensor.type == "Smoke":
        if sensor.current_value:
            print(f"[READING] {sensor.location} Smoke: DETECTED")
        else:
            print(f"[READING] {sensor.location} Smoke: CLEAR")

    elif sensor.type == "Temperature":
        if sensor.current_value < 35 or sensor.current_value > 95:
            print(f"[READING] {sensor.location} Temperature: {sensor.current_value}°F")

        elif mode == "HOME" and (sensor.current_value < 65 or sensor.current_value > 75):
            print(f"[READING] {sensor.location} Temperature: {sensor.current_value}°F")
            print(f"[NOTICE] Comfort: Temperature in {sensor.location} is outside the comfort range!")

        else:
            print(f"[READING] {sensor.location} Temperature: {sensor.current_value}°F (Normal)")

    elif sensor.type == "Motion":
        if sensor.current_value:
            print(f"[READING] {sensor.location} Motion: Activity detected")
        else:
            print(f"[READING] {sensor.location} Motion: No activity")

    elif sensor.type == "Door":
        if sensor.current_value == "OPENED":
            print(f"[READING] {sensor.location}: OPENED")

            if sensor.open_since is None:
                sensor.open_since = datetime.datetime.now()

            elif datetime.datetime.now() - sensor.open_since > datetime.timedelta(minutes=5):
                print(f"[NOTICE] {sensor.location} has been open for more than 5 minutes!")

        else:
            print(f"[READING] {sensor.location}: CLOSED")
            sensor.open_since = None

    alert_message = process_reading(sensor, mode)

    if alert_message:
        trigger_alert(alert_message)

def process_readings(sensors, mode):

    for sensor in sensors:
        sensor_update(sensor, mode)

    if motion_sensor.current_value and door_sensor.current_value == "OPENED" and mode == "AWAY":
        trigger_alert("CRITICAL SECURITY: Multiple sensors triggered - possible break-in!")

In [139]:
def run_simulation(iterations=3, system_mode="AWAY"):
    print("=== HomeGuard Security System ===")

    start_time = datetime.datetime.now().replace(second=0, microsecond=0)

    for i in range(iterations):
        current_time = start_time + datetime.timedelta(minutes=i)

        print(f"\nTime: {current_time.strftime('%H:%M:%S')}")
        print(f"Mode: {system_mode}")
        print()

        for sensor in sensors:
            sensor.update_value(generate_reading(sensor))

        process_readings(sensors, system_mode)


run_simulation(iterations=3, system_mode="AWAY")

=== HomeGuard Security System ===

Time: 12:12:00
Mode: AWAY

[READING] Living Room Motion: No activity
[READING] Front Door: CLOSED
[READING] Kitchen Temperature: 91°F (Normal)
[READING] Bedroom Smoke: CLEAR

Time: 12:13:00
Mode: AWAY

[READING] Living Room Motion: No activity
[READING] Front Door: CLOSED
[READING] Kitchen Temperature: 79°F (Normal)
[READING] Bedroom Smoke: CLEAR

Time: 12:14:00
Mode: AWAY

[READING] Living Room Motion: Activity detected
[ALERT!] 🚨 HIGH SECURITY: Motion detected in Living Room while system is in AWAY mode!
[LOG] [12:12:10] Sending notification to homeowner...
[READING] Front Door: CLOSED
[READING] Kitchen Temperature: 94°F (Normal)
[READING] Bedroom Smoke: CLEAR


In [140]:
# Step 7: Fixed Test Cases

test_cases = [
    {
        "mode": "AWAY",
        "motion": True,
        "door": "OPENED",
        "temperature": 68,
        "smoke": False
    },
    {
        "mode": "SLEEP",
        "motion": False,
        "door": "CLOSED",
        "temperature": 30,
        "smoke": True
    },
    {
        "mode": "HOME",
        "motion": False,
        "door": "CLOSED",
        "temperature": 60,
        "smoke": False
    }
]

start_time = datetime.datetime(2026, 1, 1, 14, 30, 0)

print("=== HomeGuard Security System ===")

for i, test in enumerate(test_cases):
    current_time = start_time + datetime.timedelta(minutes=i)

    print(f"\nTime: {current_time.strftime('%H:%M:%S')}")
    print(f"Mode: {test['mode']}")
    print()

    motion_sensor.update_value(test["motion"])
    door_sensor.update_value(test["door"])
    temperature_sensor.update_value(test["temperature"])
    smoke_sensor.update_value(test["smoke"])

    process_readings(sensors, test["mode"])

=== HomeGuard Security System ===

Time: 14:30:00
Mode: AWAY

[READING] Living Room Motion: Activity detected
[ALERT!] 🚨 HIGH SECURITY: Motion detected in Living Room while system is in AWAY mode!
[LOG] [12:12:10] Sending notification to homeowner...
[READING] Front Door: OPENED
[ALERT!] 🚨 HIGH SECURITY: Front Door opened while system is in AWAY mode!
[LOG] [12:12:10] Sending notification to homeowner...
[READING] Kitchen Temperature: 68°F (Normal)
[READING] Bedroom Smoke: CLEAR
[ALERT!] 🚨 CRITICAL SECURITY: Multiple sensors triggered - possible break-in!
[LOG] [12:12:10] Sending notification to homeowner...

Time: 14:31:00
Mode: SLEEP

[READING] Living Room Motion: No activity
[READING] Front Door: CLOSED
[READING] Kitchen Temperature: 30°F
[ALERT!] 🚨 WARNING: Abnormal temperature in Kitchen!
[LOG] [12:12:10] Sending notification to homeowner...
[READING] Bedroom Smoke: DETECTED
[ALERT!] 🚨 CRITICAL: Smoke detected in Bedroom!
[LOG] [12:12:10] Sending notification to homeowner...

Time